# 한국어 피싱 문자 분류 · 나이브 베이즈

**런타임 → 모두 실행**을 선택하면 데이터를 읽고, 모델 두 개를 비교한 뒤 평가 결과와 예문별 점수를 보여 줍니다. 마지막 코드 셀에서 `spam_classifier.joblib`을 내려받습니다.

기본 데이터는 [KOR_phishing_Detect-Dataset](https://github.com/Ez-Sy01/KOR_phishing_Detect-Dataset)의 한국어 정상·피싱 문자입니다. [Korean_message](https://huggingface.co/datasets/meal-bbang/Korean_message)로 바꾸려면 첫 셀의 `DATASET`을 `"lgaidataset"`으로 변경하세요. 텍스트 원본의 범위는 피싱 탐지이고, 아래 일반 광고 예문 점수는 참고용입니다.


## 1. 데이터 읽기


In [ ]:
import csv
import io
import re
import unicodedata
from collections import Counter
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen
from zipfile import ZipFile

import joblib
import numpy as np
import sklearn

DATASET = "KOR_phishing_data_raw"  # 또는 "lgaidataset"
DATASETS = {
    "KOR_phishing_data_raw": (
        "https://raw.githubusercontent.com/Ez-Sy01/KOR_phishing_Detect-Dataset/main/data/KOR_phishing_data_raw.zip",
        {"0": 0, "1": 1},
    ),
    "lgaidataset": (
        "https://huggingface.co/datasets/meal-bbang/Korean_message/resolve/main/lgaidataset.csv",
        {"1": 0, "2": 1},
    ),
}

url, label_map = DATASETS[DATASET]
csv_path = Path(f"{DATASET}.csv")
if csv_path.is_file():
    csv_bytes = csv_path.read_bytes()
else:
    try:
        with urlopen(url, timeout=25) as response:
            csv_bytes = response.read()
    except (URLError, TimeoutError):
        from google.colab import files

        print(f"{csv_path.name} 파일을 업로드하세요.")
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("CSV 파일 하나를 선택하세요.")
        csv_bytes = next(iter(uploaded.values()))

if csv_bytes.startswith(b"PK\x03\x04"):
    with ZipFile(io.BytesIO(csv_bytes)) as archive:
        members = [name for name in archive.namelist() if Path(name).name == csv_path.name]
        if len(members) != 1:
            raise ValueError(f"ZIP에서 {csv_path.name} 파일 하나를 찾을 수 있어야 합니다.")
        csv_bytes = archive.read(members[0])

try:
    raw_csv = csv_bytes.decode("utf-8-sig")
except UnicodeDecodeError:
    raw_csv = csv_bytes.decode("cp949")
reader = csv.DictReader(io.StringIO(raw_csv, newline=""))
if not {"content", "class"}.issubset(reader.fieldnames or []):
    raise ValueError("CSV에는 content(본문)와 class(라벨) 열이 있어야 합니다.")

rows = [
    (row["content"], label_map[row["class"].strip()])
    for row in reader
    if row["content"] and row["content"].strip()
    and row["class"] is not None and row["class"].strip() in label_map
]
print(f"{DATASET}: 사용 가능한 원본 {len(rows):,}건")
print("라벨 분포 (0=정상, 1=피싱):", dict(sorted(Counter(label for _, label in rows).items())))


## 2. 중복 정리와 학습·검증·평가 분리


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

def normalized_text(text):
    return " ".join(unicodedata.normalize("NFKC", text).casefold().split())


def template_key(text):
    text = re.sub(r"(?:https?://|www\.)\S+", "<url>", text)
    return re.sub(r"\d+", "0", text)


# 충돌하는 라벨이 붙은 중복 본문을 제외하고, 같은 문구는 한 번만 사용합니다.
unique = {}
for text, label in rows:
    key = normalized_text(text)
    if key not in unique:
        unique[key] = (text, label)
    elif unique[key] is not None and unique[key][1] != label:
        unique[key] = None

examples = [(text, label, template_key(key)) for key, value in unique.items()
            if value is not None for text, label in [value]]
message_texts = np.array([text for text, _, _ in examples], dtype=object)
labels = np.array([label for _, label, _ in examples])
groups = np.array([group for _, _, group in examples], dtype=object)

# 링크나 숫자만 다른 유사 문구도 같은 그룹에 넣습니다.
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
folds = [held_out for _, held_out in splitter.split(message_texts, labels, groups)]
test_indices, validation_indices = folds[:2]
train_indices = np.concatenate(folds[2:])

for name, indices in [
    ("학습", train_indices), ("모델 선택·확률 보정", validation_indices), ("최종 평가", test_indices),
]:
    counts = Counter(labels[indices].tolist())
    if set(counts) != {0, 1}:
        raise ValueError(f"{name} 구간에 정상과 피싱 라벨이 모두 필요합니다.")
    print(f"{name}: {len(indices):,}건 (정상 {counts[0]:,}, 피싱 {counts[1]:,})")


## 3. 나이브 베이즈 학습과 평가

AP는 피싱 문자를 얼마나 높은 점수로 찾아내는지 보여 줍니다. 최종 평가에서는 피싱 정밀도와 재현율도 확인하세요.


In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.frozen import FrozenEstimator
from sklearn.metrics import average_precision_score, classification_report, confusion_matrix
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.pipeline import make_pipeline

candidates = {
    "단어 빈도 + MultinomialNB": make_pipeline(CountVectorizer(), MultinomialNB()),
    "글자 묶음 TF-IDF + ComplementNB": make_pipeline(
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True),
        ComplementNB(alpha=0.5),
    ),
}

validation_scores = {}
for name, model in candidates.items():
    model.fit(message_texts[train_indices], labels[train_indices])
    scores = model.predict_proba(message_texts[validation_indices])[:, 1]
    validation_scores[name] = average_precision_score(labels[validation_indices], scores)
    print(f"검증 AP | {name} | {validation_scores[name]:.3f}")

selected_name = max(validation_scores, key=validation_scores.get)
selected = candidates[selected_name]
# 테스트 구간은 모델 선택이나 확률 보정에 사용하지 않습니다.
model = CalibratedClassifierCV(FrozenEstimator(selected), method="sigmoid")
model.fit(message_texts[validation_indices], labels[validation_indices])

test_texts = message_texts[test_indices]
test_labels = labels[test_indices]
test_scores = model.predict_proba(test_texts)[:, 1]
print(f"\n선택한 모델: {selected_name}")
print(f"최종 평가 AP: {average_precision_score(test_labels, test_scores):.3f}")
print(classification_report(test_labels, model.predict(test_texts),
                            labels=[0, 1], target_names=["정상", "피싱"], digits=3, zero_division=0))
print("혼동 행렬(행은 실제, 열은 예측, 순서는 정상·피싱)")
print(confusion_matrix(test_labels, model.predict(test_texts), labels=[0, 1]))


## 4. 예문 입력과 모델 저장


In [ ]:
examples_to_try = [
    "안녕하세요! 내일 학교 끝나고 도서관에서 만날까요?",
    "[광고] 오늘만 전 상품 할인! 무료 쿠폰을 신청하세요.",
    "[웹발신] 배송 주소 오류로 배송이 지연되었습니다. 주소를 확인해 주세요.",
]

for text, probability in zip(examples_to_try, model.predict_proba(examples_to_try)[:, 1]):
    print(f"> {text}\n[ 스팸 확률 : {probability:.1%} ]\n")

model_path = Path("spam_classifier.joblib")
joblib.dump(model, model_path, compress=3)
reloaded = joblib.load(model_path)
np.testing.assert_allclose(reloaded.predict_proba(examples_to_try),
                           model.predict_proba(examples_to_try))
print(f"저장 완료: {model_path.name} (scikit-learn {sklearn.__version__}, joblib {joblib.__version__})")


## 5. 모델 다운로드

받은 파일을 `spam_model_test.py`와 같은 폴더에 두면 기존 테스트 프로그램에서 열 수 있습니다. 저장 결과에 표시된 scikit-learn 버전으로 실행하세요.


In [ ]:
from google.colab import files

files.download(str(model_path))
